# Fast Sentiment Analysis Using Distilled Transformers
## Baseline Notebook — TF-IDF + LR & GloVe-BiLSTM

**Team:** Arwa Elgazar · Eman Elsayed · Esraa Nematalla — Deep Learning, Queen's University

This notebook builds two baselines before introducing DistilBERT:
1. **TF-IDF + Logistic Regression** — classical, fast, lightweight
2. **BiLSTM + GloVe + Self-Attention** — deep learning baseline with pretrained embeddings

> Run top-to-bottom · `SEED=42` · CPU-only · No GPU required


## §0 — Install Dependencies

Installing all required packages so the notebook runs on any machine without manual setup.


In [1]:
import subprocess, sys

# List of packages we need for the whole project
# We install them here so the notebook works out-of-the-box on any machine
PACKAGES = [
    "torch", "transformers>=4.36.0", "datasets>=2.14.0",
    "accelerate>=0.26.0", "evaluate", "scikit-learn>=1.3.0",
    "seaborn", "pandas", "numpy", "psutil", "matplotlib",
    "onnx", "onnxruntime",
]

print("Installing / verifying packages ...")
for pkg in PACKAGES:
    # Run pip install quietly (-q) and check if it succeeded
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    print(f"  {'OK' if r.returncode == 0 else 'FAILED'} {pkg}")
print("Done.\n")


Installing / verifying packages ...
  OK torch
  OK transformers>=4.36.0
  OK datasets>=2.14.0
  OK accelerate>=0.26.0
  OK evaluate
  OK scikit-learn>=1.3.0
  OK seaborn
  OK pandas
  OK numpy
  OK psutil
  OK matplotlib
  OK onnx
  OK onnxruntime
Done.



## §1 — Imports, Config & Metric Definitions

All hyperparameters live in the `CFG` dict — one place to change things, no magic numbers scattered around.

**Metrics used (binary, K=2):**

$$\text{Accuracy} = \frac{TP+TN}{N} \qquad
P_k = \frac{TP_k}{TP_k+FP_k} \qquad
R_k = \frac{TP_k}{TP_k+FN_k}$$

$$F1_k = \frac{2 P_k R_k}{P_k + R_k} \qquad
\text{Macro-F1} = \frac{1}{K}\sum_k F1_k \quad \text{(primary metric)}$$

**Latency:** median wall-clock time over 50 single-sample runs after 5 warm-up discards.


In [ ]:
# Standard library imports
import os, io, re, html, json, time, random, warnings, tracemalloc, pickle

# Data science stack
import numpy as np
import pandas as pd

# Plotting — we use Agg backend so it works on machines without a display (e.g. servers)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch — for building the BiLSTM model
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from collections import Counter  # used to build our vocabulary

# scikit-learn — for the TF-IDF + Logistic Regression baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

# HuggingFace — for loading datasets and the DistilBERT tokenizer
from datasets import load_dataset, Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback, EvalPrediction,
)

# Suppress non-critical warnings to keep output clean
warnings.filterwarnings('ignore')

#  Reproducibility 
# We fix every random seed so results are identical across runs.
# SEED=42 is set once here and reused everywhere.
SEED = 42
def set_seed(s=SEED):
    random.seed(s)          # Python built-in random
    np.random.seed(s)       # NumPy
    torch.manual_seed(s)    # PyTorch CPU ops
    os.environ['PYTHONHASHSEED'] = str(s)  # Python hash randomisation
set_seed()

# Force CPU — this whole project runs without a GPU by design
DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

# Create output directories so we can save models, results, and figures
BASE = os.path.join(os.getcwd(), 'project_upgraded')
for sub in ['data', 'models/sst2', 'models/imdb', 'models/bilstm',
            'models/onnx', 'results', 'figures']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)

# Global Config Dictionary 
# All hyperparameters are stored here so we never have magic numbers scattered around
CFG = {
    'model_name'       : 'distilbert-base-uncased',  # pretrained model for DistilBERT (used later)
    'sst2_train_size'  : None,   # None means use the FULL 67K training set
    'sst2_val_size'    : None,   # None means use all 872 validation samples
    'sst2_test_size'   : 1000,
    'sst2_max_len'     : 128,    # 128 tokens covers 100% of SST-2 sentences (median is only 9 words)
    'imdb_train_size'  : None,   # Full 25K IMDb training set
    'imdb_val_split'   : 0.20,   # 20% of train → validation set
    'imdb_max_len'     : 512,    # We tested 256 vs 512 — 512 gives +1.06pp at only +0.58ms extra
    'tfidf_ngram_sst2' : (1, 1), # Ablation shows unigrams beat bigrams on short SST-2 sentences
    'tfidf_ngram_imdb' : (1, 2), # Bigrams work better for long IMDb reviews (negation patterns)
    'tfidf_max_features': 50_000, # Keep vocabulary manageable — top 50K most frequent terms
    'logreg_C'         : 1.0,    # Regularisation strength (default, works well here)
    'logreg_max_iter'  : 1000,   # Enough iterations for lbfgs to converge
    'bert_batch_size'  : 16,
    'bert_lr'          : 2e-5,
    'bert_weight_decay': 0.01,
    'bert_warmup_ratio': 0.10,
    'bert_grad_clip'   : 1.0,
    'bert_train_epochs': 3,
    'bilstm_embed_dim' : 100,    # Match GloVe 100d vector size
    'bilstm_hidden'    : 128,    # 128 hidden units per direction → 256 for bidirectional concat
    'bilstm_layers'    : 2,      # 2-layer LSTM gives better representational depth
    'bilstm_dropout'   : 0.3,    # Dropout between layers to reduce overfitting
    'bilstm_epochs'    : 8,      # Max epochs — early stopping will kick in if needed
    'bilstm_patience'  : 3,      # Stop if F1 doesn't improve for 3 epochs in a row
    'glove_dim'        : 100,    # We use GloVe 6B 100-dimensional vectors
    'latency_warmup'   : 5,      # Discard first 5 runs — CPU needs to warm up
    'latency_runs'     : 50,     # Time 50 samples and take the median
    'ece_bins'         : 15,
    'throughput_batches': [1, 8, 16, 32, 64],
}

# Label mappings — consistent across all models
ID2LABEL = {0:'NEGATIVE', 1:'POSITIVE'}
LABEL2ID = {'NEGATIVE':0, 'POSITIVE':1}

# Print environment info so results are reproducible on other machines
import platform
import psutil
print('ENVIRONMENT')
print(f'  Python      : {platform.python_version()}')
print(f'  PyTorch     : {torch.__version__}')
import transformers
print(f'  Transformers: {transformers.__version__}')
print(f'  CPU cores   : {psutil.cpu_count(logical=False)} physical / {psutil.cpu_count()} logical')
print(f'  RAM         : {psutil.virtual_memory().total/1e9:.1f} GB')
print(f'  Device      : {DEVICE}')
print(f'  Seed        : {SEED}')


Device: cpu
ENVIRONMENT
  Python      : 3.13.9
  PyTorch     : 2.11.0+cpu
  Transformers: 5.10.2
  CPU cores   : 4 physical / 8 logical
  RAM         : 17.1 GB
  Device      : cpu
  Seed        : 42


## §2 — Load & Explore SST-2

SST-2 contains short phrase-level movie-review fragments (median **9 words**, max 52).
We use the full **67,349** training samples and the official **872-sample** validation set as our test set.

`max_length=128` covers 100% of sentences with zero truncation — confirmed by the CDF in panel (c).


In [3]:
print('Loading SST-2 (GLUE) ...')
# Load SST-2 from HuggingFace — this downloads it automatically on first run
sst2_raw = load_dataset('nyu-mll/glue', 'sst2')

def stratified_hf_subset(hf_ds, n, seed=SEED):
    """
    Create a smaller stratified subset of a HuggingFace dataset.
    If n is None or larger than the dataset, the full dataset is returned.
    Stratified sampling ensures both classes are equally represented in the subset.
    """
    df = hf_ds.to_pandas()
    if n is None or n >= len(df):
        return hf_ds  # use full dataset
    sub, _ = train_test_split(df, train_size=n, stratify=df['label'], random_state=seed)
    return HFDataset.from_pandas(sub.reset_index(drop=True))

# Since sst2_train_size=None in CFG, this just returns the full 67K training set
sst2_train_ds = stratified_hf_subset(sst2_raw['train'],      CFG['sst2_train_size'])
sst2_val_ds   = stratified_hf_subset(sst2_raw['validation'], CFG['sst2_val_size'])
sst2_test_ds  = sst2_raw['validation']  # official held-out test set (not touched during training)

print(f'  train={len(sst2_train_ds):,}  val={len(sst2_val_ds):,}  test={len(sst2_test_ds):,}')

# Print class distribution for each split to check for imbalance
for sn, ds in [('train', sst2_train_ds), ('val', sst2_val_ds), ('test', sst2_test_ds)]:
    lbl = ds['label']
    neg = sum(1 for l in lbl if l == 0)
    pos = sum(1 for l in lbl if l == 1)
    print(f'  {sn:5s}  neg={neg}  pos={pos}  ({neg/(neg+pos)*100:.1f}% neg)')

# Convert to pandas for EDA and save to disk for reference
sst2_tr_df = sst2_train_ds.to_pandas()
sst2_vl_df = sst2_val_ds.to_pandas()
sst2_te_df = sst2_test_ds.to_pandas()
sst2_tr_df.to_csv(f'{BASE}/data/sst2_train.csv', index=False)
sst2_te_df.to_csv(f'{BASE}/data/sst2_test.csv',  index=False)

# ── EDA Plots ───────────────────────────────────────────────────────────────
# We add a word count column to understand sentence length distribution
sst2_tr_df['wlen'] = sst2_tr_df['sentence'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SST-2 EDA — Full Training Set (67,349 samples)', fontweight='bold')

# Panel (a): word length histogram — shows most sentences are very short
axes[0].hist(sst2_tr_df['wlen'], bins=40, color='#3498DB', edgecolor='white', alpha=0.85)
axes[0].axvline(sst2_tr_df['wlen'].median(), color='red', ls='--', lw=2,
                label=f"Median={sst2_tr_df['wlen'].median():.0f}")
axes[0].set_title('(a) Word-Length Distribution'); axes[0].legend()

# Panel (b): class balance bar chart
sst2_tr_df['label'].value_counts().plot.bar(ax=axes[1],
    color=['#E74C3C', '#2ECC71'], edgecolor='white', alpha=0.85)
axes[1].set_title('(b) Class Distribution'); axes[1].set_xticklabels(['NEG', 'POS'], rotation=0)

# Panel (c): CDF — shows that max_len=128 covers 100% of sentences (no truncation at all)
axes[2].ecdf(sst2_tr_df['wlen'])
axes[2].axvline(128, color='red', ls='--', lw=2, label='max_len=128 (100% coverage)')
axes[2].set_title('(c) CDF — Token Coverage'); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{BASE}/figures/sst2_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Word stats: mean={sst2_tr_df['wlen'].mean():.1f}  "
      f"median={sst2_tr_df['wlen'].median():.0f}  max={sst2_tr_df['wlen'].max()}")


Loading SST-2 (GLUE) ...


  train=67,349  val=872  test=872
  train  neg=29780  pos=37569  (44.2% neg)
  val    neg=428  pos=444  (49.1% neg)
  test   neg=428  pos=444  (49.1% neg)
  Word stats: mean=9.4  median=7  max=52


## §3 — Load & Explore IMDb

IMDb has full-length movie reviews (median **230 words**, max ~2,400) — the opposite of SST-2.
We use all **25,000** training samples (80/20 train/val split) and keep the 25K test set fully held out.

`max_length=512` was chosen over 256 after ablation: it covers significantly more of each review and gives **+1.06pp** accuracy at only +0.58ms extra latency.
Raw HTML artifacts (`<br>`, `&amp;`) are cleaned before any modelling.


In [ ]:
def clean_imdb(text):
    """
    Clean raw IMDb review text.
    The dataset was scraped from the web so it contains HTML artifacts like <br />
    and HTML entities like &amp; that need to be removed before we can use it.
    Steps:
      1. Decode HTML entities (e.g. &amp; -> &)
      2. Strip all HTML tags (e.g. <br /> -> space)
      3. Lowercase everything
      4. Collapse multiple spaces into one
    """
    text = html.unescape(text)            # fix &amp; &lt; etc.
    text = re.sub(r'<[^>]+>', ' ', text)  # remove <br>, <b>, etc.
    text = text.lower()                   # lowercase for consistency
    return re.sub(r'\s+', ' ', text).strip()  # collapse whitespace

print('Loading IMDb ...')
# This loads the full dataset — 25K train, 25K test, 50K unlabelled
imdb_raw   = load_dataset('stanfordnlp/imdb')
imdb_train = imdb_raw['train'].to_pandas()
imdb_test  = imdb_raw['test'].to_pandas()
print(f'  Train: {len(imdb_train):,}  Test: {len(imdb_test):,}')

# Apply HTML cleaning to all reviews
imdb_train['clean'] = imdb_train['text'].apply(clean_imdb)
imdb_test['clean']  = imdb_test['text'].apply(clean_imdb)

# Shuffle the training data before splitting (good practice to remove any ordering bias)
imdb_sub = imdb_train.copy().sample(frac=1, random_state=SEED).reset_index(drop=True)

# Split: 80% train, 20% validation — stratified to keep class balance
imdb_X_tr, imdb_X_vl, imdb_y_tr, imdb_y_vl = train_test_split(
    imdb_sub['clean'].to_numpy(), imdb_sub['label'].to_numpy(),
    test_size=CFG['imdb_val_split'], stratify=imdb_sub['label'].to_numpy(),
    random_state=SEED)
print(f'  IMDb  train: {len(imdb_X_tr):,} | val: {len(imdb_X_vl):,}')

# Save splits to disk so we don't have to re-process later
pd.DataFrame({'text': imdb_X_tr, 'label': imdb_y_tr}).to_csv(f'{BASE}/data/imdb_train.csv', index=False)
pd.DataFrame({'text': imdb_X_vl, 'label': imdb_y_vl}).to_csv(f'{BASE}/data/imdb_val.csv', index=False)
pd.DataFrame({'text': imdb_test['clean'], 'label': imdb_test['label']}).to_csv(
    f'{BASE}/data/imdb_test_full.csv', index=False)

#  EDA Plots 
imdb_train['wlen'] = imdb_train['clean'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('IMDb EDA — Full Training Set (25,000 samples)', fontweight='bold')

# Panel (a): word length histogram — shows reviews are MUCH longer than SST-2
axes[0].hist(imdb_train['wlen'], bins=60, color='#E74C3C', edgecolor='white', alpha=0.85)
axes[0].axvline(imdb_train['wlen'].median(), color='black', ls='--', lw=2,
                label=f"Median={imdb_train['wlen'].median():.0f}")
axes[0].set_title('(a) Word-Length Distribution'); axes[0].legend(fontsize=8)

# Panel (b): class balance — perfectly 50/50
imdb_train['label'].value_counts().plot.bar(ax=axes[1],
    color=['#E74C3C', '#2ECC71'], edgecolor='white', alpha=0.85)
axes[1].set_title('(b) Class Distribution'); axes[1].set_xticklabels(['NEG', 'POS'], rotation=0)

# Panel (c): CDF comparing max_len=256 vs 512
# This is the evidence we used to justify choosing max_len=512 over 256
axes[2].ecdf(imdb_train['wlen'])
for ml, c in [(256, 'orange'), (512, 'red')]:
    pct = (imdb_train['wlen'] <= ml).mean() * 100
    axes[2].axvline(ml, color=c, ls='--', lw=2, label=f'max_len={ml} ({pct:.0f}% coverage)')
axes[2].set_title('(c) CDF — Token Coverage'); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{BASE}/figures/imdb_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Word stats: mean={imdb_train['wlen'].mean():.0f}  "
      f"median={imdb_train['wlen'].median():.0f}  max={imdb_train['wlen'].max()}")


Loading IMDb ...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

  Train: 25,000  Test: 25,000
  IMDb  train: 20,000 | val: 5,000
  Word stats: mean=231  median=173  max=2459


## §4 — Preprocessing Pipelines

Two separate pipelines — applying the same preprocessing to both models would hurt the transformer.

| Pipeline | Steps | Why |
|---|---|---|
| Classical / BiLSTM | lowercase + collapse whitespace | TF-IDF and BiLSTM vocab are case-insensitive by design |
| DistilBERT | raw text → WordPiece tokenizer | model was pretrained on raw text; our lowercasing would be redundant |

Stop words and punctuation are **kept** — negations like *"not good"* carry sentiment signal.


In [5]:
def clean_classical(text):
    """
    Minimal normalisation for TF-IDF and BiLSTM pipelines.
    We only lowercase and collapse whitespace — nothing more.
    We intentionally keep punctuation and stop words because:
      - Negation words (not, never, n't) carry sentiment signal
      - Punctuation like '!' can indicate strong sentiment
    """
    return re.sub(r'\s+', ' ', str(text).lower().strip())

# Apply classical cleaning to all SST-2 splits
X_sst2_tr = [clean_classical(s) for s in sst2_tr_df['sentence']]
X_sst2_vl = [clean_classical(s) for s in sst2_vl_df['sentence']]
X_sst2_te = [clean_classical(s) for s in sst2_te_df['sentence']]
y_sst2_tr = list(sst2_tr_df['label'])
y_sst2_vl = list(sst2_vl_df['label'])
y_sst2_te = list(sst2_te_df['label'])

# IMDb data was already cleaned by clean_imdb() in the previous cell
X_imdb_tr = list(imdb_X_tr); y_imdb_tr = list(imdb_y_tr)
X_imdb_vl = list(imdb_X_vl); y_imdb_vl = list(imdb_y_vl)

#  DistilBERT tokenisation 
# We also prepare HuggingFace tokenised datasets here.
# These are needed for the DistilBERT fine-tuning in the main notebook.
print(f'Loading tokenizer: {CFG["model_name"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

def make_hf_ds(texts, labels, col, max_len):
    """
    Tokenise a list of texts and wrap them in a HuggingFace Dataset.
    - truncation=True: cut sequences longer than max_len
    - padding=False: we use dynamic padding later via DataCollatorWithPadding
      (this is more efficient than padding everything to max_len upfront)
    - batch_size=512: tokenise in chunks of 512 for memory efficiency
    """
    ds = HFDataset.from_dict({col: list(texts), 'label': list(labels)})
    ds = ds.map(lambda b: tokenizer(b[col], truncation=True,
                max_length=max_len, padding=False), batched=True, batch_size=512)
    ds = ds.remove_columns([col])  # remove the original text column — we only need token IDs
    ds.set_format('torch')         # return PyTorch tensors instead of lists
    return ds

# Tokenise SST-2 (max_len=128 covers 100% of sentences)
print('Tokenising SST-2 ...')
hf_sst2_tr = make_hf_ds(sst2_tr_df['sentence'], y_sst2_tr, 'sentence', CFG['sst2_max_len'])
hf_sst2_vl = make_hf_ds(sst2_vl_df['sentence'], y_sst2_vl, 'sentence', CFG['sst2_max_len'])
hf_sst2_te = make_hf_ds(sst2_te_df['sentence'], y_sst2_te, 'sentence', CFG['sst2_max_len'])

# Tokenise IMDb (max_len=512 — our Tier-1 upgrade over 256)
print(f'Tokenising IMDb (max_len={CFG["imdb_max_len"]}) ...')
hf_imdb_tr = make_hf_ds(imdb_X_tr, imdb_y_tr, 'text', CFG['imdb_max_len'])
hf_imdb_vl = make_hf_ds(imdb_X_vl, imdb_y_vl, 'text', CFG['imdb_max_len'])
print('Preprocessing complete.')


Loading tokenizer: distilbert-base-uncased ...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Tokenising IMDb (max_len=512) ...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Preprocessing complete.


## §5 — Classical Baseline: TF-IDF + Logistic Regression

TF-IDF converts each sentence into a weighted bag-of-words vector, then Logistic Regression classifies it. Fast, interpretable, and surprisingly strong.

**N-gram ablation on SST-2** (sentences average 9 words — bigrams are too sparse):

| n-gram | Acc | F1 |
|---|---|---|
| **(1,1) unigrams** | **82.34%** | **0.8228** ← selected |
| (1,2) bigrams | 80.73% | 0.8066 |
| (1,3) trigrams | 81.08% | 0.8101 |

For IMDb we keep `(1,2)` — longer reviews benefit from bigrams capturing phrases like *"not good"*.


In [6]:
def build_lr(ngram_range, seed=42):
    """
    Build a TF-IDF + Logistic Regression pipeline.
    We wrap them in a sklearn Pipeline so fit/predict works on raw text.

    TF-IDF settings:
      - sublinear_tf=True: apply log(1+tf) instead of raw tf to dampen
        the effect of very frequent words (e.g. 'the', 'is')
      - strip_accents='unicode': normalise accented characters
      - token_pattern=r'\w{2,}': only keep tokens with 2+ characters
        (filters out single-character noise like 's' or 'a')
      - min_df=2: ignore words that appear in fewer than 2 documents
        (removes very rare typos and noise)
    """
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=CFG['tfidf_max_features'],  # cap vocab at 50K terms
            sublinear_tf=True,
            strip_accents='unicode',
            analyzer='word',
            token_pattern=r'\w{2,}',
            min_df=2)),
        ('clf', LogisticRegression(
            C=CFG['logreg_C'],          # C=1.0 is the default — works well here
            solver='lbfgs',             # lbfgs is efficient for large feature spaces
            max_iter=CFG['logreg_max_iter'],  # 1000 is enough for convergence
            random_state=seed)),
    ])

def _state_dict_size_mb(model):
    """Helper to measure model size in MB by serialising to a byte buffer."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / (1024**2)

def eval_classical(pipe, X_val, y_val, X_bench, name):
    """
    Evaluate a classical sklearn pipeline and measure latency + model size.
    Returns a dict with all metrics so we can compare models in a unified table later.
    """
    # Get predictions on the validation set
    y_pred = pipe.predict(X_val)
    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average='macro')
    prec = precision_score(y_val, y_pred, average='macro')
    rec  = recall_score(y_val, y_pred, average='macro')

    print(f'  [{name}]  acc={acc*100:.2f}%  F1={f1:.4f}')
    print(classification_report(y_val, y_pred, target_names=['NEGATIVE', 'POSITIVE']))

    # Latency measurement — time per single sample (simulates real-world inference)
    # We do 5 warm-up runs first to let the CPU cache warm up
    for _ in range(CFG['latency_warmup']):
        pipe.predict([X_bench[0]])
    times = []
    for t in X_bench[:CFG['latency_runs']]:  # time 50 samples
        t0 = time.perf_counter()
        pipe.predict([t])
        times.append((time.perf_counter() - t0) * 1000)  # convert to ms

    lat_med = float(np.median(times))    # we report median not mean (more robust to outliers)
    lat_p95 = float(np.percentile(times, 95))  # p95 shows worst-case tail latency
    size_mb = len(pickle.dumps(pipe)) / 1e6   # serialise to bytes and measure size

    print(f'  Latency: {np.mean(times):.3f}+/-{np.std(times):.3f} ms | median={lat_med:.3f} | p95={lat_p95:.3f}')
    print(f'  Size   : {size_mb:.3f} MB')

    return {'accuracy': acc, 'f1_macro': f1, 'precision': prec, 'recall': rec,
            'lat_med_ms': lat_med, 'lat_p95_ms': lat_p95, 'size_mb': size_mb,
            'labels': y_pred.tolist() if hasattr(y_pred, 'tolist') else list(y_pred)}

# N-gram ablation on SST-2 
# We test 3 n-gram ranges to find which works best for short SST-2 sentences
print('N-gram ablation on SST-2 ...')
ngram_results = []
for ng in [(1, 1), (1, 2), (1, 3)]:
    set_seed()  # reset seed so each run is comparable
    p = build_lr(ng)
    p.fit(X_sst2_tr, y_sst2_tr)
    yp  = p.predict(X_sst2_vl)
    acc = accuracy_score(y_sst2_vl, yp)
    f1  = f1_score(y_sst2_vl, yp, average='macro')
    mark = ' <- SELECTED' if ng == CFG['tfidf_ngram_sst2'] else ''
    print(f'  ngram={ng}  acc={acc*100:.2f}%  F1={f1:.4f}{mark}')
    ngram_results.append({'ngram': str(ng), 'accuracy': acc, 'f1': f1})

# Save ablation results to CSV for the report
pd.DataFrame(ngram_results).to_csv(f'{BASE}/results/ngram_ablation_sst2.csv', index=False)

#  Train final SST-2 baseline with the winning n-gram setting 
print('\nTraining SST-2 LR+TF-IDF with selected n-gram (1,1) ...')
set_seed()
lr_sst2 = build_lr(CFG['tfidf_ngram_sst2'])
t0 = time.perf_counter()
lr_sst2.fit(X_sst2_tr, y_sst2_tr)
print(f'  Training time: {time.perf_counter()-t0:.2f}s')
lr_sst2_metrics = eval_classical(lr_sst2, X_sst2_vl, y_sst2_vl, X_sst2_vl, 'SST-2 LR')
# Save model to disk so we can load it later without retraining
with open(f'{BASE}/models/sst2/logreg.pkl', 'wb') as f:
    pickle.dump(lr_sst2, f)

#  Train IMDb baseline with bigrams 
print('\nTraining IMDb LR+TF-IDF with ngram (1,2) ...')
set_seed()
lr_imdb = build_lr(CFG['tfidf_ngram_imdb'])
t0 = time.perf_counter()
lr_imdb.fit(X_imdb_tr, y_imdb_tr)
print(f'  Training time: {time.perf_counter()-t0:.2f}s')
lr_imdb_metrics = eval_classical(lr_imdb, X_imdb_vl, y_imdb_vl, X_imdb_vl, 'IMDb LR')
with open(f'{BASE}/models/imdb/logreg.pkl', 'wb') as f:
    pickle.dump(lr_imdb, f)


N-gram ablation on SST-2 ...
  ngram=(1, 1)  acc=82.34%  F1=0.8228 <- SELECTED
  ngram=(1, 2)  acc=80.39%  F1=0.8032
  ngram=(1, 3)  acc=81.31%  F1=0.8124

Training SST-2 LR+TF-IDF with selected n-gram (1,1) ...
  Training time: 1.76s
  [SST-2 LR]  acc=82.34%  F1=0.8228
              precision    recall  f1-score   support

    NEGATIVE       0.85      0.78      0.81       428
    POSITIVE       0.80      0.86      0.83       444

    accuracy                           0.82       872
   macro avg       0.83      0.82      0.82       872
weighted avg       0.83      0.82      0.82       872

  Latency: 1.368+/-0.630 ms | median=1.254 | p95=2.761
  Size   : 0.605 MB

Training IMDb LR+TF-IDF with ngram (1,2) ...
  Training time: 21.74s
  [IMDb LR]  acc=88.88%  F1=0.8888
              precision    recall  f1-score   support

    NEGATIVE       0.89      0.88      0.89      2500
    POSITIVE       0.88      0.89      0.89      2500

    accuracy                           0.89      5000
   m

## §6 — Download GloVe Embeddings

We use **GloVe 6B 100d** (Stanford NLP) to initialise the BiLSTM embedding layer instead of random weights.
This gave us **+4.35pp** on SST-2 compared to Xavier random initialisation in earlier experiments.

The zip (~862 MB) is downloaded once, we extract only the 100d file, then delete the zip.
About **65.9%** of our vocabulary words are found in GloVe; the rest get small random initialisations.


In [7]:
import os, zipfile, urllib.request

GLOVE_PATH = 'glove.6B.100d.txt'   # the file we actually need
GLOVE_URL  = 'https://nlp.stanford.edu/data/glove.6B.zip'
GLOVE_ZIP  = 'glove.6B.zip'        # temporary zip file

def _progress_hook(count, block_size, total_size):
    """Print a simple download progress bar to the console."""
    if total_size > 0:
        pct = min(count * block_size / total_size * 100, 100)
        bar = int(pct // 2)
        print(f'\r  [{"#"*bar}{"."}*(50-bar)] {pct:.1f}%', end='', flush=True)

def download_glove():
    """
    Download and extract GloVe 6B 100d embeddings.
    Uses only Python stdlib (urllib) — no wget or curl needed.
    The function is idempotent: if the file already exists, it does nothing.
    """
    # Skip download if we already have the extracted file
    if os.path.exists(GLOVE_PATH):
        size_mb = os.path.getsize(GLOVE_PATH) / 1e6
        print(f'GloVe already present: {GLOVE_PATH} ({size_mb:.0f} MB)')
        return

    # Step 1: Download the zip file if it's not already here
    if not os.path.exists(GLOVE_ZIP):
        print(f'Downloading GloVe 6B from Stanford (~862 MB) ...')
        print('  This takes a few minutes depending on your internet speed.')
        try:
            urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP, reporthook=_progress_hook)
            print()  # newline after progress bar finishes
            print(f'  Download complete: {os.path.getsize(GLOVE_ZIP)/1e6:.0f} MB')
        except Exception as e:
            print(f'\n  Download failed: {e}')
            print('  Manual fix: download from https://nlp.stanford.edu/data/glove.6B.zip')
            print('  and place glove.6B.100d.txt in your working directory.')
            raise
    else:
        print(f'  Zip already present: {GLOVE_ZIP}')

    # Step 2: Extract only the 100d file (we don't need 50d, 200d, or 300d)
    print('  Extracting glove.6B.100d.txt ...')
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract('glove.6B.100d.txt', '.')
    os.remove(GLOVE_ZIP)  # delete the zip to free ~862 MB of disk space
    print(f'  Done. File: {GLOVE_PATH} ({os.path.getsize(GLOVE_PATH)/1e6:.0f} MB)')

download_glove()

def load_glove_embeddings(glove_path, vocab, embed_dim=100):
    """
    Load pretrained GloVe vectors into a (vocab_size, embed_dim) numpy matrix.

    For each word in our vocabulary:
      - If it's in GloVe: use its pretrained 100-dim vector
      - If it's not in GloVe (OOV): keep a small random init from Uniform(-0.1, 0.1)
      - The PAD token (index 0) is always set to all-zeros

    Returns:
      matrix: torch.Tensor of shape (vocab_size, embed_dim)
      coverage_pct: float — percentage of vocab words found in GloVe
    """
    # Start with small random values for all words (OOV fallback)
    matrix = __import__('numpy').random.uniform(
        -0.1, 0.1, (len(vocab), embed_dim)).astype('float32')
    matrix[0] = 0.0  # PAD token must be a zero vector (no signal)

    # Read through GloVe file and replace random vectors with pretrained ones
    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word  = parts[0]
            if word in vocab.word2idx:  # only load vectors for words we actually need
                matrix[vocab.word2idx[word]] = __import__('numpy').array(
                    parts[1:], dtype='float32')
                found += 1

    cov = found / len(vocab) * 100
    print(f'  GloVe coverage: {found}/{len(vocab)} tokens ({cov:.1f}%)')
    return __import__('torch').tensor(matrix, dtype=__import__('torch').float32), cov

print('GloVe loader ready.')


  This takes a few minutes depending on your internet speed.
  [##################################################.*(50-bar)] 100.0%
  Download complete: 862 MB
  Extracting glove.6B.100d.txt ...
  Done. File: glove.6B.100d.txt (347 MB)
GloVe loader ready.


## §7 — Second Baseline: BiLSTM + GloVe + Self-Attention

Architecture: `GloVe(100d) → Dropout → BiLSTM(128h, 2-layer) → Self-Attention → Linear(2)`

**Bidirectional:** reads the sequence both ways so no signal is missed at either end.

**Self-attention:** instead of using only the last hidden state, it computes a weighted average over all timesteps — learning to focus on the most sentiment-relevant words.

**Phased embedding training:**
- Epoch 1: embeddings **frozen** — lets the LSTM stabilise before touching GloVe weights
- Epoch 2+: embeddings **unfrozen** at 10× lower lr (`1e-4`) to fine-tune without overwriting pretrained knowledge


In [8]:
RUN_BILSTM = True  # set to False to skip training and load saved weights instead

# ── Vocabulary class 
class Vocabulary:
    """
    Simple word-to-index vocabulary built from training text.
    Two special tokens are always present:
      - <PAD> (index 0): used to pad shorter sequences in a batch
      - <UNK> (index 1): used for words not seen during vocabulary building
    Only words appearing at least min_freq times are included (filters noise).
    """
    PAD, UNK = '<PAD>', '<UNK>'

    def __init__(self, max_size=30_000):
        self.max_size = max_size
        # Pre-populate with the two special tokens
        self.word2idx = {self.PAD: 0, self.UNK: 1}
        self.idx2word = {0: self.PAD, 1: self.UNK}

    def build(self, texts, min_freq=2):
        """Count all words across all texts and keep the top max_size by frequency."""
        counter = Counter()
        for t in texts:
            counter.update(str(t).lower().split())
        # Add words in frequency order, stopping at max_size
        for word, freq in counter.most_common(self.max_size - 2):
            if freq >= min_freq:  # skip very rare words (likely noise)
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, text, max_len):
        """
        Convert a text string to a fixed-length list of integer token IDs.
        - Truncate to max_len if too long
        - Pad with 0 (PAD index) if too short
        - Unknown words map to index 1 (UNK)
        """
        tokens = str(text).lower().split()[:max_len]
        ids    = [self.word2idx.get(t, 1) for t in tokens]  # 1 = UNK for missing words
        ids   += [0] * (max_len - len(ids))                 # pad to max_len
        return ids

    def __len__(self):
        return len(self.word2idx)


class SeqDataset(TorchDataset):
    """PyTorch Dataset that pre-encodes all texts at construction time."""
    def __init__(self, texts, labels, vocab, max_len):
        # Encode everything upfront — faster than encoding on-the-fly during training
        self.data = [
            (torch.tensor(vocab.encode(t, max_len), dtype=torch.long),
             torch.tensor(int(l), dtype=torch.long))
            for t, l in zip(texts, labels)
        ]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]


# ── Self-Attention layer 
class SelfAttention(nn.Module):
    """
    Additive self-attention over BiLSTM hidden states.
    For each time step t, compute a scalar score:
      score_t = W * h_t   (learnable weight W)
    Then take a weighted average of all hidden states:
      context = sum_t( softmax(score_t) * h_t )
    This gives us a single vector that "pays attention" to the most relevant tokens.
    Padding positions are masked out (set to -inf before softmax) so they don't contribute.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # Single linear layer: maps from 2*hidden_dim (bidirectional) to 1 score per token
        self.attn = nn.Linear(hidden_dim * 2, 1, bias=False)

    def forward(self, lstm_out, mask=None):
        # lstm_out shape: (batch, seq_len, 2*hidden_dim)
        scores = self.attn(lstm_out).squeeze(-1)       # (batch, seq_len) — one score per token
        if mask is not None:
            # Mask out PAD positions so they don't affect the weighted average
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)        # (batch, seq_len) — attention weights
        # Weighted sum across time dimension
        context = (weights.unsqueeze(-1) * lstm_out).sum(dim=1)  # (batch, 2*hidden_dim)
        return context, weights


# BiLSTM + Attention model 
class BiLSTMAttention(nn.Module):
    """
    Full architecture:
      Embedding -> Dropout -> BiLSTM -> SelfAttention -> Dropout -> Linear classifier

    The embedding layer is initialised with GloVe vectors.
    The BiLSTM processes the sequence in both directions.
    The attention layer collapses the sequence into a single context vector.
    The classifier maps that vector to 2 class logits.
    """
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128,
                 num_layers=2, dropout=0.3, padding_idx=0):
        super().__init__()
        # Embedding layer — will be initialised with GloVe weights after creation
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        # Bidirectional LSTM — outputs 2*hidden_dim features per time step
        self.bilstm     = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                                  batch_first=True, bidirectional=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
        self.attention  = SelfAttention(hidden_dim)
        self.dropout    = nn.Dropout(dropout)
        # Final classifier: 2*hidden_dim (bidirectional concat) -> 2 classes
        self.classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):
        # x shape: (batch, seq_len) — integer token IDs
        mask = (x != 0)                              # True where token is not PAD
        emb  = self.dropout(self.embedding(x))       # (batch, seq_len, embed_dim)
        out, _ = self.bilstm(emb)                    # (batch, seq_len, 2*hidden_dim)
        ctx, attn_w = self.attention(out, mask)      # ctx: (batch, 2*hidden_dim)
        return self.classifier(self.dropout(ctx)), attn_w  # logits + attention weights

    def freeze_embeddings(self, freeze=True):
        """Toggle whether the embedding weights are updated during backprop."""
        self.embedding.weight.requires_grad = not freeze


#  Build shared vocabulary from BOTH datasets 
# We build one vocabulary covering both SST-2 and IMDb so the embedding matrix
# is shared — this is more efficient than having two separate models
print('Building shared vocabulary ...')
vocab = Vocabulary(max_size=30_000)
vocab.build(X_sst2_tr + X_imdb_tr, min_freq=2)  # min_freq=2 removes hapax legomena
print(f'  Vocabulary size: {len(vocab):,}')

# Load GloVe and create the pretrained embedding matrix
print('Loading GloVe into embedding matrix ...')
glove_matrix, glove_coverage = load_glove_embeddings(
    GLOVE_PATH, vocab, embed_dim=CFG['glove_dim'])

def build_bilstm_model():
    """Create a fresh BiLSTM model and initialise its embeddings with GloVe."""
    model = BiLSTMAttention(
        vocab_size  = len(vocab),
        embed_dim   = CFG['bilstm_embed_dim'],
        hidden_dim  = CFG['bilstm_hidden'],
        num_layers  = CFG['bilstm_layers'],
        dropout     = CFG['bilstm_dropout'],
    )
    # Copy GloVe vectors into the embedding layer
    model.embedding.weight.data.copy_(glove_matrix)
    model.embedding.weight.data[0] = 0  # PAD stays all-zeros no matter what
    return model.to(DEVICE)

# Quick parameter count to report in the paper
n_params = sum(p.numel() for p in build_bilstm_model().parameters())
print(f'BiLSTM+Attention params: {n_params:,} ({n_params/1e6:.2f}M)')


Building shared vocabulary ...
  Vocabulary size: 30,000
Loading GloVe into embedding matrix ...
  GloVe coverage: 19768/30000 tokens (65.9%)
BiLSTM+Attention params: 3,631,554 (3.63M)


In [9]:
def train_bilstm(model, train_texts, train_labels, val_texts, val_labels,
                 max_len, dataset_name, bilstm_lr=1e-3, embed_lr=1e-4):
    """
    Train the BiLSTM with a phased embedding strategy:
      - Epoch 1:   embeddings frozen, only LSTM + classifier learn (lr=1e-3)
      - Epoch 2+:  embeddings unfrozen with 10x lower lr (1e-4) to fine-tune GloVe

    Why phased? If we unfreeze GloVe from the start, the LSTM's random weights
    generate noisy gradients that corrupt the pretrained embeddings before they
    can be useful. Freezing epoch 1 lets the LSTM stabilise first.

    Early stopping: saves the best checkpoint by val F1 and stops if no improvement
    for 'bilstm_patience' consecutive epochs.
    """
    # Wrap data in Dataset and DataLoader for batched training
    train_ds = SeqDataset(train_texts, train_labels, vocab, max_len)
    val_ds   = SeqDataset(val_texts,   val_labels,   vocab, max_len)
    train_dl = DataLoader(train_ds, batch_size=64,  shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=0)

    criterion = nn.CrossEntropyLoss()  # standard loss for multi-class classification

    # Phase 1 optimizer: only train parameters where requires_grad=True
    # (embeddings are frozen, so this only touches LSTM + classifier)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=bilstm_lr)

    best_f1, patience_count = 0.0, 0
    history = []  # track metrics per epoch for plotting later
    print(f'\nTraining BiLSTM+GloVe+Attention on {dataset_name} ...')

    for epoch in range(1, CFG['bilstm_epochs'] + 1):

        # ── Phase transition 
        if epoch == 1:
            # Freeze embeddings for the first epoch
            model.freeze_embeddings(True)
            print('  [Epoch 1] Embeddings FROZEN -- LSTM+Classifier learning')
        elif epoch == 2:
            # Unfreeze embeddings from epoch 2 with a much smaller learning rate
            model.freeze_embeddings(False)
            # Rebuild optimizer with per-layer learning rates
            optimizer = torch.optim.Adam([
                {'params': model.embedding.parameters(),  'lr': embed_lr},    # slow for embeddings
                {'params': model.bilstm.parameters(),     'lr': bilstm_lr},   # normal for LSTM
                {'params': model.attention.parameters(),  'lr': bilstm_lr},   # normal for attention
                {'params': model.classifier.parameters(), 'lr': bilstm_lr},   # normal for head
            ])
            print('  [Epoch 2+] Embeddings UNFROZEN (lr=1e-4)')

        # ── Training loop 
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, _ = model(xb)          # forward pass (ignore attention weights during training)
            loss = criterion(logits, yb)   # cross-entropy loss
            optimizer.zero_grad()          # clear gradients from previous step
            loss.backward()                # backprop
            optimizer.step()               # update weights
            epoch_loss += loss.item()

        # ── Validation 
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():  # no grad needed for evaluation — saves memory and time
            for xb, yb in val_dl:
                logits, _ = model(xb.to(DEVICE))
                all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
                all_true.extend(yb.tolist())

        val_acc  = accuracy_score(all_true, all_preds)
        val_f1   = f1_score(all_true, all_preds, average='macro')
        avg_loss = epoch_loss / len(train_dl)
        print(f'  Epoch {epoch}/{CFG["bilstm_epochs"]}  '
              f'loss={avg_loss:.4f}  val_acc={val_acc*100:.2f}%  val_f1={val_f1:.4f}')
        history.append({'epoch': epoch, 'loss': avg_loss, 'val_acc': val_acc, 'val_f1': val_f1})

        #  Early stopping 
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_count = 0
            # Save the best model checkpoint
            torch.save(model.state_dict(), f'{BASE}/models/bilstm/best_{dataset_name}.pt')
        else:
            patience_count += 1
            if patience_count >= CFG['bilstm_patience']:
                print(f'  Early stopping at epoch {epoch} (best val_f1={best_f1:.4f})')
                break

    # Restore the best checkpoint before returning
    model.load_state_dict(torch.load(f'{BASE}/models/bilstm/best_{dataset_name}.pt',
                                     map_location=DEVICE, weights_only=True))
    print(f'  Best val_f1={best_f1:.4f}')
    return model, history


#  Train on SST-2
# max_len=64 is enough for SST-2 (median 9 words, we want a bit of headroom)
MAX_LEN_BILSTM_SST2 = 64
set_seed()  # reset seed before each model for full reproducibility
bilstm_sst2 = build_bilstm_model()
bilstm_sst2, hist_sst2 = train_bilstm(
    bilstm_sst2, X_sst2_tr, y_sst2_tr, X_sst2_vl, y_sst2_vl,
    MAX_LEN_BILSTM_SST2, 'sst2')

#  Evaluate SST-2 BiLSTM
val_ds_s2 = SeqDataset(X_sst2_vl, y_sst2_vl, vocab, MAX_LEN_BILSTM_SST2)
val_dl_s2 = DataLoader(val_ds_s2, batch_size=256, shuffle=False)
bilstm_sst2.eval()
preds_bl_s2 = []
with torch.no_grad():
    for xb, _ in val_dl_s2:
        # argmax over class logits to get predicted label
        preds_bl_s2.extend(torch.argmax(bilstm_sst2(xb.to(DEVICE))[0], dim=1).cpu().tolist())

acc_bl_s2 = accuracy_score(y_sst2_vl, preds_bl_s2)
f1_bl_s2  = f1_score(y_sst2_vl, preds_bl_s2, average='macro')
print(classification_report(y_sst2_vl, preds_bl_s2, target_names=['NEGATIVE', 'POSITIVE']))

#  SST-2 Latency & size measurement 
# 5 warm-up runs first, then time 50 individual samples
for _ in range(CFG['latency_warmup']):
    enc = torch.tensor([vocab.encode(X_sst2_vl[0], MAX_LEN_BILSTM_SST2)]).to(DEVICE)
    bilstm_sst2(enc)
bl_times_s2 = []
with torch.no_grad():
    for t in X_sst2_vl[:CFG['latency_runs']]:
        enc = torch.tensor([vocab.encode(t, MAX_LEN_BILSTM_SST2)]).to(DEVICE)
        t0  = time.perf_counter()
        bilstm_sst2(enc)
        bl_times_s2.append((time.perf_counter() - t0) * 1000)  # ms

bl_lat_s2 = float(np.median(bl_times_s2))
buf = io.BytesIO()
torch.save(bilstm_sst2.state_dict(), buf)
bl_size_s2 = buf.tell() / (1024**2)  # MB
print(f'  BiLSTM SST-2  acc={acc_bl_s2*100:.2f}%  F1={f1_bl_s2:.4f}  '
      f'lat={bl_lat_s2:.3f}ms  size={bl_size_s2:.2f}MB')

# Store metrics in a dict for the unified comparison table later
bilstm_sst2_metrics = {'accuracy': acc_bl_s2, 'f1_macro': f1_bl_s2,
                        'lat_med_ms': bl_lat_s2, 'size_mb': bl_size_s2}


#  Train on IMDb 
# max_len=256 for IMDb (reviews are much longer — 256 is the sweet spot for BiLSTM speed)
MAX_LEN_BILSTM_IMDB = 256
set_seed()
bilstm_imdb = build_bilstm_model()
bilstm_imdb, hist_imdb = train_bilstm(
    bilstm_imdb, X_imdb_tr, y_imdb_tr, X_imdb_vl, y_imdb_vl,
    MAX_LEN_BILSTM_IMDB, 'imdb')

# ── Evaluate IMDb BiLSTM ─────────────────────────────────────────────────────
val_ds_im = SeqDataset(X_imdb_vl, y_imdb_vl, vocab, MAX_LEN_BILSTM_IMDB)
val_dl_im = DataLoader(val_ds_im, batch_size=64, shuffle=False)
bilstm_imdb.eval()
preds_bl_im = []
with torch.no_grad():
    for xb, _ in val_dl_im:
        preds_bl_im.extend(torch.argmax(bilstm_imdb(xb.to(DEVICE))[0], dim=1).cpu().tolist())

acc_bl_im = accuracy_score(y_imdb_vl, preds_bl_im)
f1_bl_im  = f1_score(y_imdb_vl, preds_bl_im, average='macro')
print(classification_report(y_imdb_vl, preds_bl_im, target_names=['NEGATIVE', 'POSITIVE']))

#  IMDb Latency & size 
for _ in range(CFG['latency_warmup']):
    enc = torch.tensor([vocab.encode(X_imdb_vl[0], MAX_LEN_BILSTM_IMDB)]).to(DEVICE)
    bilstm_imdb(enc)
bl_times_im = []
with torch.no_grad():
    for t in X_imdb_vl[:CFG['latency_runs']]:
        enc = torch.tensor([vocab.encode(t, MAX_LEN_BILSTM_IMDB)]).to(DEVICE)
        t0  = time.perf_counter()
        bilstm_imdb(enc)
        bl_times_im.append((time.perf_counter() - t0) * 1000)

bl_lat_im = float(np.median(bl_times_im))
buf = io.BytesIO()
torch.save(bilstm_imdb.state_dict(), buf)
bl_size_im = buf.tell() / (1024**2)
print(f'  BiLSTM IMDb   acc={acc_bl_im*100:.2f}%  F1={f1_bl_im:.4f}  '
      f'lat={bl_lat_im:.3f}ms  size={bl_size_im:.2f}MB')
bilstm_imdb_metrics = {'accuracy': acc_bl_im, 'f1_macro': f1_bl_im,
                        'lat_med_ms': bl_lat_im, 'size_mb': bl_size_im}

#  Plot training curves for both datasets 
# The dashed vertical line at epoch 2 marks where embeddings were unfrozen
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('GloVe+Attention-BiLSTM Training Curves', fontweight='bold')

for ax, hist, ds in [(axes[0], hist_sst2, 'SST-2'), (axes[1], hist_imdb, 'IMDb')]:
    epochs = [h['epoch'] for h in hist]
    f1s    = [h['val_f1'] for h in hist]
    losses = [h['loss'] for h in hist]
    ax2 = ax.twinx()  # second y-axis for F1 on the same plot
    ax.plot(epochs, losses, 'b-o', ms=4, label='Train Loss')
    ax2.plot(epochs, [f * 100 for f in f1s], 'r-s', ms=4, label='Val F1 (%)')
    ax2.axvline(2, color='gray', ls=':', lw=1.5, label='Embeddings unfrozen')  # phase transition marker
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='b')
    ax2.set_ylabel('Val F1 (%)', color='r')
    ax.set_title(f'{ds} -- Peak F1={max(f1s)*100:.2f}%')
    # Combine legends from both axes
    lines1, lab1 = ax.get_legend_handles_labels()
    lines2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, lab1 + lab2, fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(f'{BASE}/figures/bilstm_training.png', dpi=150, bbox_inches='tight')
plt.show()



Training BiLSTM+GloVe+Attention on sst2 ...
  [Epoch 1] Embeddings FROZEN -- LSTM+Classifier learning
  Epoch 1/8  loss=0.4357  val_acc=81.88%  val_f1=0.8188
  [Epoch 2+] Embeddings UNFROZEN (lr=1e-4)
  Epoch 2/8  loss=0.3572  val_acc=83.49%  val_f1=0.8348
  Epoch 3/8  loss=0.3061  val_acc=84.29%  val_f1=0.8429
  Epoch 4/8  loss=0.2685  val_acc=87.04%  val_f1=0.8704
  Epoch 5/8  loss=0.2390  val_acc=85.67%  val_f1=0.8562
  Epoch 6/8  loss=0.2161  val_acc=87.50%  val_f1=0.8750
  Epoch 7/8  loss=0.1986  val_acc=86.47%  val_f1=0.8643
  Epoch 8/8  loss=0.1849  val_acc=87.84%  val_f1=0.8784
  Best val_f1=0.8784
              precision    recall  f1-score   support

    NEGATIVE       0.87      0.88      0.88       428
    POSITIVE       0.88      0.88      0.88       444

    accuracy                           0.88       872
   macro avg       0.88      0.88      0.88       872
weighted avg       0.88      0.88      0.88       872

  BiLSTM SST-2  acc=87.84%  F1=0.8784  lat=4.756ms  size=1